# 24 · Survival Analysis / Time-to-Event

No todos los targets son `sí/no` o una cantidad. A veces importa **cuándo** ocurre un evento: abandono, falla, reingreso, churn, término de beneficio o tiempo hasta una intervención. Survival Analysis trata explícitamente observaciones censuradas.

## Objetivos
- Entender censoring y por qué una regresión común es incorrecta.
- Interpretar función de supervivencia $S(t)$ y hazard $h(t)$.
- Estimar Kaplan–Meier.
- Comparar curvas con log-rank.
- Ajustar Cox Proportional Hazards.
- Evaluar con concordance index.
- Introducir Random Survival Forest y survival ML.


In [ ]:
!pip -q install lifelines
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
SEED=42; rng=np.random.default_rng(SEED); n=1800
age=rng.normal(45,12,n); group=rng.binomial(1,.45,n); risk=np.exp(.025*(age-45)+.55*group)
true_time=rng.exponential(scale=36/risk); censor_time=rng.uniform(8,55,n); observed=np.minimum(true_time,censor_time); event=(true_time<=censor_time).astype(int)
df=pd.DataFrame({'time':observed,'event':event,'age':age,'group':group}); print(df.event.mean()); df.head()

## 1. Censoring
Si una persona sigue activa al final del estudio, sabemos que su tiempo al evento es **mayor** que el tiempo observado, no que el evento nunca ocurrirá. Tirar esos casos o tratarlos como negativos sesga el análisis.

Censoring derecho es el caso más común. Survival clásico suele asumir censoring independiente condicionado en covariables relevantes.


## 2. Survival y hazard
$S(t)=P(T>t)$ es la probabilidad de permanecer sin evento más allá de $t$.

El hazard $h(t)$ representa una tasa instantánea de evento condicionada a haber sobrevivido hasta $t$. No es exactamente una probabilidad.


In [ ]:
km=KaplanMeierFitter(); km.fit(df.time,event_observed=df.event,label='global'); ax=km.plot_survival_function(); ax.set(xlabel='tiempo',ylabel='S(t)',title='Kaplan–Meier'); plt.show(); print('mediana de supervivencia',km.median_survival_time_)

## 3. Comparar grupos
Kaplan–Meier puede mostrar curvas por segmento. El log-rank test compara la experiencia de supervivencia global de dos grupos, bajo supuestos que conviene revisar.


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for g in [0,1]:
 sub=df[df.group==g]; KaplanMeierFitter().fit(sub.time,sub.event,label=f'group={g}').plot_survival_function(ax=ax)
plt.show()
a=df[df.group==0]; b=df[df.group==1]; print(logrank_test(a.time,b.time,event_observed_A=a.event,event_observed_B=b.event).summary)

## 4. Cox Proportional Hazards
Cox modela:
$$h(t|x)=h_0(t)\exp(\beta^T x)$$

$e^{\beta_j}$ se interpreta como **hazard ratio**. HR=1.5 implica una tasa instantánea 50% mayor, no '50% más probabilidad total'.

El supuesto de proportional hazards significa que el hazard ratio entre perfiles es aproximadamente constante en el tiempo.


In [ ]:
cox=CoxPHFitter().fit(df,duration_col='time',event_col='event'); cox.print_summary(); cox.plot(); plt.show(); cox.check_assumptions(df,p_value_threshold=.05,show_plots=False)

## 5. Evaluación
El concordance index mide si pares de individuos están correctamente ordenados por riesgo. 0.5 ≈ azar, 1.0 perfecto. También existen time-dependent AUC y Brier score integrados.


In [ ]:
risk_score=cox.predict_partial_hazard(df).to_numpy().ravel(); print('C-index',concordance_index(df.time,-risk_score,df.event))

## 6. ML de supervivencia
Extensiones:
- Random Survival Forest;
- Gradient Boosting Survival;
- XGBoost `survival:cox` / AFT;
- DeepSurv y neural survival models;
- competing risks;
- recurrent events;
- multi-state models.

### Competing risks
Si pueden ocurrir eventos mutuamente excluyentes (p.ej. salida por distintas causas), censurar una causa como si fuera censoring independiente puede ser incorrecto. Se usan cumulative incidence / Fine–Gray.

## Casos de uso
- tiempo hasta deserción o egreso;
- churn;
- tiempo hasta falla de equipo;
- duración de procesos;
- reingreso;
- tiempo hasta adopción de un servicio.

## Errores comunes
- convertir survival en clasificación binaria ignorando tiempo;
- eliminar censurados;
- interpretar hazard ratio como riesgo absoluto;
- no revisar proportional hazards;
- usar un cutoff temporal arbitrario sin justificarlo.

## Ejercicios
1. Simula censoring 20%, 50% y 80% y observa estabilidad.
2. Añade una covariable con efecto dependiente del tiempo y detecta violación PH.
3. Construye curvas KM por cuartiles de edad.
4. Instala `scikit-survival` y prueba Random Survival Forest.
5. Calcula survival probability individual a t=12 y t=24.
6. Investiga competing risks y cumulative incidence.
